# 04 — Autoencoder e modelos clássicos do PTB-XL

**DCA3606 — Inteligência Artificial (UFRN).** Esta etapa compara Random Forest sobre sinais normalizados vetorizados com o mesmo Random Forest sobre embeddings de um autoencoder 1D. O autoencoder é treinado **antes**, de forma não supervisionada, e seu encoder é congelado para a extração. As tarefas usam os mesmos alvos e partições das etapas 2 e 3.

O notebook inicia configurado para o **treinamento completo no Colab**, conforme os caminhos abaixo. Fora do Colab, esse modo é bloqueado. Para uma verificação local, altere temporariamente `MODO_TESTE=True`; o teste rápido foi executado separadamente e os resultados aparecem ao final. O fold 10 permanece reservado para a avaliação final futura.

## 1. Configuração de caminhos e execução

O Colab monta o Drive e pode copiar os cinco arquivos preparados para `/content/ptbxl_preparado`, mantendo checkpoints e resultados no Drive. A cópia local acelera a leitura da matriz de aproximadamente 0,974 GiB. Todos os arquivos são conferidos antes da cópia e novamente antes dos experimentos. Em VS Code, o modo de teste usa a raiz local do projeto. Dependências: `torch`, `numpy`, `pandas`, `matplotlib`, `scikit-learn`, `joblib` e `ipykernel`. Se faltarem no Colab, execute `%pip install torch numpy pandas matplotlib scikit-learn joblib` antes desta célula.

In [ ]:
from pathlib import Path
import shutil
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display

MODO_TESTE = False
EXECUTAR_TREINO_COMPLETO = True

RAIZ_PROJETO_COLAB = Path('/content/drive/MyDrive/IA-2026')

PASTA_DADOS_PREPARADOS_COLAB = (
    RAIZ_PROJETO_COLAB / 'resultados' / 'preprocessamento'
)

PASTA_SAIDA_COLAB = (
    RAIZ_PROJETO_COLAB / 'resultados' / 'autoencoder_classicos'
)

COPIAR_DADOS_PARA_CONTENT = True
SEMENTE = 3606

try:
    import google.colab
    EM_COLAB = True
except ModuleNotFoundError:
    EM_COLAB = False
if not MODO_TESTE and (not EM_COLAB or not EXECUTAR_TREINO_COMPLETO):
    raise RuntimeError('Treinamento completo permitido apenas no Colab com EXECUTAR_TREINO_COMPLETO=True.')
if EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    RAIZ_PROJETO = RAIZ_PROJETO_COLAB
    PASTA_DADOS_ORIGEM = PASTA_DADOS_PREPARADOS_COLAB
    PASTA_SAIDA_BASE = PASTA_SAIDA_COLAB
else:
    RAIZ_PROJETO = Path.cwd()
    PASTA_DADOS_ORIGEM = RAIZ_PROJETO / 'resultados' / 'preprocessamento'
    PASTA_SAIDA_BASE = RAIZ_PROJETO / 'resultados' / 'autoencoder_classicos'
if not (RAIZ_PROJETO / 'src' / 'autoencoder_classicos.py').is_file():
    raise FileNotFoundError(f'Módulo src/autoencoder_classicos.py ausente em {RAIZ_PROJETO}')
sys.path.insert(0, str(RAIZ_PROJETO))

from src.autoencoder_classicos import (
    ARQUIVOS_PREPARADOS, AutoencoderECG1D, ConfigAE, ConfigRF,
    conferir_arquivos_preparados, conferir_correspondencia,
    contar_parametros, extrair_embeddings, grafico_reconstrucoes,
    preparar_pastas, salvar_comparacoes, selecionar_amostras,
    treinar_autoencoder, treinar_random_forest
)

print('Arquivos na origem (bytes):')
display(pd.Series(conferir_arquivos_preparados(PASTA_DADOS_ORIGEM), name='bytes').to_frame())
PASTA_DADOS = PASTA_DADOS_ORIGEM
if EM_COLAB and COPIAR_DADOS_PARA_CONTENT:
    PASTA_DADOS = Path('/content/ptbxl_preparado')
    PASTA_DADOS.mkdir(parents=True, exist_ok=True)
    for nome in ARQUIVOS_PREPARADOS:
        origem = PASTA_DADOS_ORIGEM / nome
        destino = PASTA_DADOS / nome
        if not destino.is_file() or destino.stat().st_size != origem.stat().st_size:
            try:
                shutil.copy2(origem, destino)
            except OSError as exc:
                raise RuntimeError(f'Falha ao copiar {nome} para /content; confira espaço livre e acesso ao Drive.') from exc
    conferir_arquivos_preparados(PASTA_DADOS)
PASTA_SAIDA = PASTA_SAIDA_BASE / ('teste_rapido_v1' if MODO_TESTE else 'treino_completo_v1')
pastas = preparar_pastas(PASTA_SAIDA)
dispositivo = torch.device('cpu' if MODO_TESTE else ('cuda' if torch.cuda.is_available() else 'cpu'))
if MODO_TESTE:
    torch.set_num_threads(2)
torch.manual_seed(SEMENTE)
np.random.seed(SEMENTE)
print('Modo teste:', MODO_TESTE, '| Colab:', EM_COLAB, '| dispositivo AE:', dispositivo)
print('Random Forest: CPU | dados:', PASTA_DADOS, '| resultados:', PASTA_SAIDA)
print('PyTorch:', torch.__version__)

## 2. Conferência de alvos, partições e correspondência dos sinais

A validação reutiliza `verificar_artefatos` da etapa 3 e confere novamente cada linha dos CSVs de tarefa com `manifesto.csv`: `ecg_id`, paciente, fold, partição, índice do sinal e alvo. A matriz binária tem forma `(21.789, 1.000, 12)` em `float32`; cada `linha_sinal` aponta a um único ECG. A amostra do autoencoder contém **ECGs únicos**, sem rótulos, apenas dos folds 1–9. No modo rápido, ela é sorteada do manifesto sem usar idade ou diagnóstico.

In [ ]:
parametros, tabelas = conferir_correspondencia(PASTA_DADOS)
shape = tuple(parametros['shape_sinais'])
unicos_ae, amostras_rf = selecionar_amostras(tabelas, MODO_TESTE, SEMENTE)
assert not unicos_ae.particao.eq('teste').any()
assert unicos_ae.ecg_id.is_unique and unicos_ae.linha_sinal.is_unique
assert tabelas['manifesto'].groupby('patient_id').particao.nunique().max() == 1
print('Matriz normalizada:', shape, 'float32')
print('AE — exames únicos:', unicos_ae.particao.value_counts().to_dict())
for tarefa, tabela in amostras_rf.items():
    print(tarefa, '— mesmos exemplos para as duas representações:',
          tabela.particao.value_counts().to_dict())
    assert not tabela.particao.eq('teste').any()
    assert tabela.ecg_id.isin(unicos_ae.ecg_id).all()
print('Fold 10 reservado; nenhum paciente cruza partições.')

## 3. Autoencoder 1D sem rótulos

O encoder reduz os 12 canais ao longo do tempo por três convoluções com stride 2 e projeta o resultado em um vetor latente de dimensão configurável. O decoder faz a operação inversa com três convoluções transpostas e saída **linear**, reconstruindo `(12, 1.000)`. O espaço latente padrão é de **64 valores**, contra **12.000 atributos** da entrada (redução de 187,5 vezes).

A perda é `SmoothL1Loss(beta=1)`: quadrática para erros pequenos e linear para erros grandes. Como a EDA sinalizou amplitudes extremas em alguns ECGs, essa perda limita a influência de erros muito altos sem descartar registros. Ela compara somente sinal reconstruído e sinal original normalizado; **idade e diagnóstico não entram no Dataset nem na perda**. O melhor modelo é escolhido pela menor perda de reconstrução na validação (fold 9).

In [ ]:
DIMENSAO_LATENTE = 64
config_ae = ConfigAE(
    dimensao_latente=DIMENSAO_LATENTE,
    epocas=1 if MODO_TESTE else 50,
    tamanho_lote=8 if MODO_TESTE else 64,
    taxa_aprendizado=1e-3,
    decaimento_pesos=1e-4,
    paciencia=2 if MODO_TESTE else 8,
    delta_minimo=1e-4,
    semente=SEMENTE,
    trabalhadores=0 if MODO_TESTE else 2,
    modo_teste=MODO_TESTE,
)
modelo_inspecao = AutoencoderECG1D(DIMENSAO_LATENTE)
print('Parâmetros treináveis do AE:', contar_parametros(modelo_inspecao))
print('Dimensão original → latente:', 12000, '→', DIMENSAO_LATENTE)
del modelo_inspecao
resumo_ae = treinar_autoencoder(PASTA_DADOS, pastas, unicos_ae, shape, config_ae, dispositivo)
display(pd.DataFrame([resumo_ae]))

## 4. Embeddings do melhor encoder

O checkpoint de menor perda de validação é carregado, o modelo entra em `eval()` e todos os parâmetros são congelados. A extração percorre somente os exames únicos de treino e validação, sem embaralhar. `indice_embeddings.csv` relaciona cada linha de `representacoes.npy` com `ecg_id`, `linha_sinal` e partição. A extração é reutilizada após desconexão somente quando o índice e o hash do checkpoint coincidem.

In [ ]:
caminho_embeddings, indice_embeddings = extrair_embeddings(
    PASTA_DADOS, pastas, unicos_ae, shape, DIMENSAO_LATENTE, dispositivo,
    tamanho_lote=16 if MODO_TESTE else 128,
    trabalhadores=0 if MODO_TESTE else 2,
)
embeddings = np.load(caminho_embeddings, mmap_mode='r')
assert embeddings.shape == (len(unicos_ae), DIMENSAO_LATENTE)
assert indice_embeddings.ecg_id.is_unique
assert indice_embeddings.linha_embedding.tolist() == list(range(len(indice_embeddings)))
assert not indice_embeddings.particao.eq('teste').any()
print('Embeddings:', embeddings.shape, embeddings.dtype)
print('Primeiras correspondências:')
display(indice_embeddings.head())
del embeddings
ids_reconstrucoes = grafico_reconstrucoes(
    PASTA_DADOS, pastas, unicos_ae, shape, DIMENSAO_LATENTE, dispositivo, SEMENTE)
print('ECGs de treino usados no gráfico real × reconstruído:', ids_reconstrucoes)

## 5. Random Forest com entrada original e com embeddings

Para cada tarefa, os **mesmos registros ordenados** de treino e validação são entregues às duas representações. Os hiperparâmetros são fixados antes dos resultados e iguais entre as representações: `n_estimators=50`, `max_depth=14`, `min_samples_leaf=4`, `max_features='sqrt'`, `n_jobs=2`, `bootstrap=True` no modo completo. Isso limita profundidade, tamanho e paralelismo dos modelos diante de 12.000 atributos, sem remover atributos nem ECGs. O mesmo `random_state` é usado. **Não há busca de hiperparâmetros nesta etapa.**

Os modelos clássicos usam CPU. A entrada original é materializada apenas para a tarefa e partição em uso; após cada ajuste, as matrizes e o modelo são liberados. Se faltar RAM, o código informa o problema. Uma nova configuração conservadora deve valer para **as duas representações**, em outra pasta de rodada, preservando todos os exames e atributos.

In [ ]:
config_rf = ConfigRF(
    n_estimators=3 if MODO_TESTE else 50,
    max_depth=4 if MODO_TESTE else 14,
    min_samples_leaf=1 if MODO_TESTE else 4,
    max_features='sqrt',
    n_jobs=1 if MODO_TESTE else 2,
    random_state=SEMENTE,
    bootstrap=True,
    modo_teste=MODO_TESTE,
)
print('Hiperparâmetros do Random Forest (iguais nas duas representações):', config_rf)
resultados_rf = []
for tarefa in ('classificacao', 'regressao'):
    tabela = amostras_rf[tarefa]
    for representacao in ('original', 'embedding'):
        resultado = treinar_random_forest(
            PASTA_DADOS, pastas, tarefa, representacao, tabela, shape,
            config_rf, caminho_embeddings, indice_embeddings)
        resultados_rf.append(resultado)
salvar_comparacoes(pastas, resultados_rf)
print('Quatro Random Forests concluídos ou recuperados dos arquivos persistidos.')

## 6. Métricas de validação e auditoria final

A classificação registra acurácia, precisão e recall da classe **Alterado**, macro-F1, métricas macro e matriz de confusão. A regressão registra MAE, MSE, RMSE e R² em **anos**. Tempo de treinamento e de inferência na validação é medido para cada Random Forest. As tabelas abaixo comparam as duas representações; nenhuma métrica do fold 10 é calculada aqui.

In [ ]:
for tarefa in ('classificacao', 'regressao'):
    comparacao = pd.read_csv(pastas['metricas'] / f'comparacao_{tarefa}.csv')
    colunas = (['representacao', 'dimensao_entrada', 'ecgs_treino', 'ecgs_validacao',
                'acuracia', 'precisao', 'recall', 'f1_macro',
                'segundos_treinamento', 'segundos_inferencia_validacao']
               if tarefa == 'classificacao' else
               ['representacao', 'dimensao_entrada', 'ecgs_treino', 'ecgs_validacao',
                'mae', 'mse', 'rmse', 'r2',
                'segundos_treinamento', 'segundos_inferencia_validacao'])
    print('Comparação:', tarefa)
    display(comparacao[colunas])
    assert comparacao.assinatura_amostra.nunique() == 1
    assert comparacao.ecgs_treino.nunique() == 1
    assert comparacao.ecgs_validacao.nunique() == 1
    if tarefa == 'classificacao':
        for representacao in ('original', 'embedding'):
            print('Matriz de confusão:', representacao)
            display(pd.read_csv(pastas['metricas'] / f'matriz_confusao_{representacao}.csv'))
print('Validações concluídas: encoder sem rótulos, correspondência dos embeddings, '
      'mesmos ECGs nos RFs, fold 10 reservado e artefatos persistidos.')
import json
config_ae_salva = json.loads((pastas['configuracoes'] / 'autoencoder.json').read_text(encoding='utf-8'))
assert config_ae_salva['sem_rotulos'] is True
assert config_ae_salva['particoes_usadas'] == ['treino', 'validacao']
assert (pastas['checkpoints'] / 'autoencoder_melhor.pt').is_file()
assert (pastas['checkpoints'] / 'autoencoder_ultimo.pt').is_file()
assert (pastas['embeddings'] / 'representacoes.npy').is_file()
assert (pastas['graficos'] / 'perda_reconstrucao.png').is_file()
assert (pastas['graficos'] / 'exemplos_reconstrucao.png').is_file()
for tarefa in ('classificacao', 'regressao'):
    configs = [json.loads((pastas['configuracoes'] / f'random_forest_{tarefa}_{rep}.json').read_text(encoding='utf-8'))
               for rep in ('original', 'embedding')]
    assert configs[0]['hiperparametros'] == configs[1]['hiperparametros']
    assert configs[0]['assinatura_amostra'] == configs[1]['assinatura_amostra']
    for rep in ('original', 'embedding'):
        assert (pastas['modelos'] / f'random_forest_{tarefa}_{rep}.joblib').is_file()
        assert (pastas['metricas'] / f'random_forest_{tarefa}_{rep}.json').is_file()
print('Auditoria de arquivos concluída: AE sem rótulos, RFs comparáveis e resultados persistidos.')

## Verificação local realizada

Uma cópia temporária deste notebook foi executada com `MODO_TESTE=True` na CPU. O autoencoder treinou **uma época** com **128 ECGs únicos de treino** e **64 de validação**, usando somente sinais e identificadores; possui **2.145.036 parâmetros treináveis**. Foram gerados **192 embeddings de 64 dimensões**. Os quatro Random Forests de teste usaram, em cada tarefa, **32 ECGs de treino** e **16 de validação**, com os mesmos exemplos e hiperparâmetros nas duas representações. Checkpoints, modelos `joblib`, métricas e gráficos foram gravados em `resultados/autoencoder_classicos/teste_rapido_v1/`. A retomada do autoencoder, dos embeddings e de um Random Forest foi verificada sem repetir treinamento. As métricas dessa amostra pequena **não são resultados científicos**; a reconstrução após uma época ainda é preliminar.

## Execução completa no Google Colab

1. Confirme no Drive `/content/drive/MyDrive/IA-2026/` o notebook `04_autoencoder_modelos_classicos.ipynb`, `src/__init__.py`, `src/deep_learning.py`, `src/autoencoder_classicos.py` e os cinco arquivos de `resultados/preprocessamento/`: `normalizacao.json`, `manifesto.csv`, `classificacao.csv`, `regressao.csv`, `sinais_normalizados.float32.bin`. Os resultados anteriores em `resultados/deep_learning/treino_completo_v1/` permanecem intactos e não são necessários para executar esta etapa.
2. Abra este notebook no Colab, selecione GPU em **Ambiente de execução → Alterar tipo de ambiente de execução** e confirme as dependências. Os caminhos e flags na primeira célula já apontam para `IA-2026` e para o modo completo (`MODO_TESTE=False`, `EXECUTAR_TREINO_COMPLETO=True`).
3. Execute as células na ordem. O Drive será montado, os arquivos preparados serão verificados e copiados para `/content/ptbxl_preparado`, e os resultados persistirão em `resultados/autoencoder_classicos/treino_completo_v1/` com subpastas `checkpoints/`, `embeddings/`, `metricas/`, `graficos/`, `configuracoes/` e `modelos/`.
4. Após uma desconexão, execute novamente com a **mesma configuração**. O autoencoder retoma de `autoencoder_ultimo.pt`; embeddings e Random Forests concluídos são reutilizados após conferência dos metadados. Para modificar hiperparâmetros, escolha uma nova pasta de rodada.

O modo completo só inicia dentro do Colab. A avaliação do fold 10 será implementada na próxima etapa.